タイタニック号の生存者を回帰モデルを用いて予測するプログラム

ランダムフォレスト回帰

決定木回帰

GDBT

サポートベクター回帰

がそれぞれ使える。

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


titanic dataを読み込む

titanic dataは、train.csvとtest.csvがある。

train.csvはsurviveのカラムが記載されている。

test.csvはsurviveは記載されていない。test.csvはsurviveを予測するため。

In [2]:
import numpy as np
import pandas as pd

#
# titanic dataを置いたgoogle drive上のpathを記載してください。この直下に.csvファイルがあることを想定しています。
#
dir_path = '/content/drive/MyDrive/IPUT/人工知能/予測/data/titanic/'

#
# 学習データの読み込み(pd.read_csvを使い、表形式でデータの読み込みを行う)
#
df = pd.read_csv(dir_path + 'train.csv')

#
# テストデータの読み込み。テストデータはtrainデータの欠損値を埋める時に用いる
#
test_df = pd.read_csv(dir_path + 'test.csv')

dataの欠損値を補完する必要があります。

titanic dataには欠損値がたくさんあるので、何らかの補完をしておかないと学習ができないから。


In [3]:
print("titanic data の欠損値")

print(df.isnull().sum())

titanic data の欠損値
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [4]:
#
# 年齢の平均値と中央値を確認するときにはコメント外してちょんまげ
#
# print(df.Age.mean())
# print(df.Age.median())

# 学習データとテストデータを連結する
all_df = pd.concat([df, test_df], ignore_index=True)
# print(fill_df)

#
# 学習データとテストデータが連結されていることを確認。
# 中央値は全体の値から計算したいのでこの手法をとる
#
print(all_df.shape)

#
# 念の為、dfをfill_dfにコピー
#
fill_df = df.copy()

#
# 年齢の中央値を全てのデータから計算
#
age_median=all_df.Age.median()

#
# 年齢の欠損値を、計算しておいた中央値で補完する
# ここの年齢補完、家族情報（family name, SibSp, Parch）を使う手段もあるので、よくデータを眺めてみてください。
#
fill_df.Age = fill_df.Age.fillna(age_median)

#
# Embarked 欠損値の補完：どちらもS（サザンプトン港）としてしまう。
# ここも同じファミリーは同じ場所から乗り込んだと考えるのが妥当。よく調べてからの方が良いと思います。
#
fill_df.Embarked = fill_df.Embarked.fillna('S')

#
# 乗船した港の欠損値を再度確認する
#
print("欠損値の確認")
print(fill_df.isnull().sum())
# fill_df.head()

(1309, 12)
欠損値の確認
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64


In [5]:
#
# train_test_splitをインポート。便利！！
#
from sklearn.model_selection import train_test_split

#
# テストデータのラベルとして、y_trainにSurvivedを挿入する
#
y_train = fill_df["Survived"]

#
# x_trainから使わないカラムを除く。
# 使わないカラム： Survived（y_trainとしたので不要）, Name, Ticket, Cabin
# cabin, ticket に部屋の区画が記載されているものがあります。参考になるかもしれません。
# Name はファミリーを予測できます。
#
x_train = fill_df.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin'])

#
# 確認したい時はコメント外して
#
# print(x_train)
# print(y_train)


#
# Sex, Embarkedをホットエンコードに変換する
# 機械学習で学習できるようにするため
#
x_train = pd.get_dummies(x_train, columns=['Sex', 'Embarked'], prefix='HE')

#
# x_train, x_testを8:2に分割する。
#
x_train, x_test, y_train, y_test = train_test_split(x_train, y_train, test_size=0.2)

#
# 一旦確認
#
print(x_train.head())

     PassengerId  Pclass   Age  SibSp  Parch      Fare  HE_female  HE_male  \
737          738       1  35.0      0      0  512.3292      False     True   
121          122       3  28.0      0      0    8.0500      False     True   
690          691       1  31.0      1      0   57.0000      False     True   
508          509       3  28.0      0      0   22.5250      False     True   
839          840       1  28.0      0      0   29.7000      False     True   

      HE_C   HE_Q   HE_S  
737   True  False  False  
121  False  False   True  
690  False  False   True  
508  False  False   True  
839   True  False  False  


ランダムフォレスト回帰

In [39]:
# ライブラリのインポート
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
#
# ランダムフォレスト回帰モデルを作る
#
model = RandomForestRegressor()

#
# ランダムフォレスト回帰モデルを学習
#
model.fit(x_train, y_train)

#
# 作成したランダムフォレスト回帰モデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 1. 0. 1. 1. 1. 0. 0. 0. 1. 0. 1. 0.
 1. 0. 0. 1. 1. 0. 1. 0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0.
 1. 0. 0. 1. 1. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0.
 0. 1. 1. 0. 1. 1. 1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0.
 1. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0.
 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
推定率= 83.80 [%]


RMSE : Root Mean Squre Errorの計算

In [38]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))



学習データのRMSE(train)=0.1435
テストデータのRMSE(train)=0.3472


決定木回帰

In [41]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor()
model.fit(x_train, y_train)

#
# 作成した決定木回帰モデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[1. 1. 0. 0. 1. 0. 0. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1. 0. 0. 0. 1. 1. 0. 0.
 1. 0. 0. 1. 0. 0. 1. 0. 1. 1. 0. 0. 1. 1. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0.
 1. 0. 0. 1. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0.
 0. 1. 1. 1. 1. 1. 1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0.
 1. 1. 0. 1. 1. 0. 0. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0.
 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0.
 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 1.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
推定率= 77.65 [%]


In [42]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))


学習データのRMSE(train)=0.0000
テストデータのRMSE(train)=0.4727


GBDT

In [43]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor()

model.fit(x_train, y_train)

#
# 作成したGDBTモデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0.
 1. 0. 0. 1. 0. 0. 1. 0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0.
 0. 0. 0. 1. 1. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 1. 0.
 0. 1. 1. 0. 1. 1. 1. 0. 1. 0. 0. 0. 1. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0.
 1. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
推定率= 84.92 [%]


In [44]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))


学習データのRMSE(train)=0.2843
テストデータのRMSE(train)=0.3602


ガウスカーネルのサポートベクター回帰

In [45]:
from sklearn.svm import SVR

model = SVR()

model.fit(x_train, y_train)

#
# 作成したGDBTモデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
推定率= 68.16 [%]


In [ ]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))